# Comparación global de StyleBot
Este notebook integra los resultados definitivos de modelos clásicos, CNN y aprendizaje por refuerzo.


## Criterio de comparación
Cada componente realiza una tarea distinta. Por eso, varianza explicada, NMI, Precision@5, F1 macro y tasa de éxito se interpretan dentro de su propio contexto.


In [1]:
from pathlib import Path
import pandas as pd
import matplotlib.pyplot as plt

# Datos definitivos reunidos de los resultados de Entregas 1, 2 y 3.
rows = [
  {
    "componente": "Características",
    "modelo": "PCA (50 componentes)",
    "objetivo": "Reducir las 784 variables de cada imagen",
    "conjunto": "Entrenamiento",
    "metrica": "Varianza explicada",
    "valor": 0.863,
    "resultado": "86.3%",
    "interpretacion": "Conserva gran parte de la información con 50 componentes.",
    "limitacion": "Reduce información visual y no clasifica por sí solo.",
    "comparar_con": "PCA de este mismo proyecto",
    "fuente": "documentacion/APORTE_AGRUPAMIENTO_RECOMENDADOR.md"
  },
  {
    "componente": "No supervisado",
    "modelo": "K-Means (k=10)",
    "objetivo": "Agrupar prendas por similitud visual",
    "conjunto": "Validación",
    "metrica": "NMI",
    "valor": 0.5121,
    "resultado": "0.5121",
    "interpretacion": "Encuentra estructura parcial; las prendas superiores se traslapan.",
    "limitacion": "Los grupos no equivalen exactamente a las etiquetas reales.",
    "comparar_con": "Otros valores de k y métricas de agrupamiento",
    "fuente": "resultados/tablas/kmeans_metricas.csv"
  },
  {
    "componente": "Recomendación",
    "modelo": "PCA 50 + coseno",
    "objetivo": "Recomendar prendas visualmente similares",
    "conjunto": "Validación",
    "metrica": "Precision@5",
    "valor": 0.8321,
    "resultado": "0.8321",
    "interpretacion": "Fue la variante de recomendación con mayor precisión en validación.",
    "limitacion": "Confunde prendas de silueta parecida, como camisa y camiseta.",
    "comparar_con": "Otras variantes del recomendador",
    "fuente": "resultados/tablas/recomendador_comparacion_validacion.csv"
  },
  {
    "componente": "Clasificación",
    "modelo": "CNN Arquitectura C",
    "objetivo": "Reconocer la categoría de una prenda",
    "conjunto": "Prueba",
    "metrica": "F1 macro",
    "valor": 0.9259,
    "resultado": "0.9259",
    "interpretacion": "Fue el clasificador final con desempeño alto y equilibrado entre clases.",
    "limitacion": "Puede confundir clases visualmente semejantes.",
    "comparar_con": "Arquitecturas CNN y MobileNetV2",
    "fuente": "resultados/entrega2/tabla_evaluacion_final_prueba.csv"
  },
  {
    "componente": "Aprendizaje por refuerzo",
    "modelo": "Agente aleatorio",
    "objetivo": "Línea base para acomodar prendas",
    "conjunto": "50 semillas (1000–1049)",
    "metrica": "Tasa de éxito de prendas",
    "valor": 0.016,
    "resultado": "1.6%",
    "interpretacion": "No aprende una política útil y llega al máximo de pasos.",
    "limitacion": "Decide sin experiencia ni optimización.",
    "comparar_con": "Q-Learning y Dyna-Q",
    "fuente": "resultados/entrega3/qlearning_dynaq_resumen.csv"
  },
  {
    "componente": "Aprendizaje por refuerzo",
    "modelo": "Q-Learning A",
    "objetivo": "Acomodar prendas correctamente",
    "conjunto": "50 semillas (1000–1049)",
    "metrica": "Tasa de éxito de prendas",
    "valor": 0.924,
    "resultado": "92.4%",
    "interpretacion": "Aprende una política efectiva y supera ampliamente a la línea base.",
    "limitacion": "Requiere 15,000 episodios de entrenamiento.",
    "comparar_con": "Agentes de RL en el mismo entorno",
    "fuente": "resultados/entrega3/qlearning_dynaq_resumen.csv"
  },
  {
    "componente": "Aprendizaje por refuerzo",
    "modelo": "Dyna-Q A (n=30)",
    "objetivo": "Acomodar prendas correctamente",
    "conjunto": "50 semillas (1000–1049)",
    "metrica": "Tasa de éxito de prendas",
    "valor": 0.936,
    "resultado": "93.6%",
    "interpretacion": "Fue el mejor agente de RL: mayor éxito y menor cantidad de pasos.",
    "limitacion": "Necesita un modelo del entorno para planificar experiencias simuladas.",
    "comparar_con": "Agentes de RL en el mismo entorno",
    "fuente": "resultados/entrega3/qlearning_dynaq_resumen.csv"
  }
]
df = pd.DataFrame(rows)

def encontrar_repositorio():
    for candidate in [Path.cwd(), *Path.cwd().parents]:
        if (candidate / "resultados").exists() and (candidate / "notebooks").exists():
            return candidate
    raise FileNotFoundError("Ejecuta el notebook desde la carpeta raíz de StyleBot.")

repo = encontrar_repositorio()
salida = repo / "resultados" / "entrega4"
salida.mkdir(parents=True, exist_ok=True)
df.to_csv(salida / "tabla_comparacion_global.csv", index=False, encoding="utf-8-sig")
df[["componente", "modelo", "metrica", "resultado", "interpretacion", "limitacion"]]


FileNotFoundError: Ejecuta el notebook desde la carpeta raíz de StyleBot.

In [ ]:
fig, axes = plt.subplots(2, 1, figsize=(14, 9), constrained_layout=True)

analitica = df.iloc[:4]
axes[0].bar(analitica["modelo"], analitica["valor"], color="#247ba0")
axes[0].set_ylim(0, 1)
axes[0].set_title("Métricas de percepción y análisis (interpretar por separado)")
axes[0].set_ylabel("Valor de la métrica")
axes[0].tick_params(axis="x", rotation=18)
for i, row in analitica.reset_index(drop=True).iterrows():
    axes[0].text(i, row["valor"] + 0.03, row["resultado"], ha="center")

rl = df.iloc[4:]
axes[1].bar(rl["modelo"], rl["valor"], color="#2a9d8f")
axes[1].set_ylim(0, 1)
axes[1].set_title("Aprendizaje por refuerzo: tasa de éxito de prendas")
axes[1].set_ylabel("Tasa de éxito")
for i, row in rl.reset_index(drop=True).iterrows():
    axes[1].text(i, row["valor"] + 0.03, row["resultado"], ha="center")

fig.savefig(salida / "grafica_comparacion_global.png", dpi=180, bbox_inches="tight")
plt.show()


## Conclusión
Dyna-Q fue el mejor agente para tomar decisiones de almacenamiento. La CNN Arquitectura C fue el modelo final para clasificación. PCA, K-Means y el recomendador complementan el sistema con reducción de dimensión, agrupamiento y sugerencias de prendas similares.


In [ ]:
print("Conclusión: Dyna-Q fue el mejor agente de RL para acomodar prendas. La CNN Arquitectura C fue el clasificador final para reconocerlas.")
print("Las métricas se comparan dentro de cada componente, no entre componentes que realizan tareas distintas.")
